In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats
from scipy.stats import ks_2samp
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

from matplotlib import font_manager
import matplotlib as mpl

arial_path = "/media/scratch/fy2306/tools/fonts"
font_files = font_manager.findSystemFonts(fontpaths=arial_path)

for file in font_files:
    font_manager.fontManager.addfont(file)
    
mpl.rcParams['font.family'] = 'Arial'

In [ ]:
def conservation_lfc_cdf(
	target_strand,
	cons_subset,
	conservation_method,
	conservation_threshold,
	title=None,
	score_smoothing=False, abs_LFC=False):

	score_path = f"/media/scratch/fy2306/projects/base_editing/data/conservation/chr19_rps19.{conservation_method}100way.wigFix"

	df = pd.read_csv(
		"/media/scratch/fy2306/projects/base_editing/data/PMID_38889719/Yao_2024_S2S3_Cas9lib.txt",
		sep = "\t"
	)
	df = df[df['Target region'] != 'Non-targeting control']
	edited_pos_chr_col = 'Expected cleavage site1' # 1-based
	if target_strand != "both":
		df = df[df['Targeting strand'] == target_strand]
	else:
		df = df.groupby([edited_pos_chr_col, 'Target region'], as_index=False)['Average log2(CRISPR score)2'].mean()

	# process conservation
	conservation_scores = pd.read_csv(score_path, sep="\t", header=None, names=['chrom', edited_pos_chr_col, 'score'])
	conservation_scores[edited_pos_chr_col] = conservation_scores[edited_pos_chr_col].astype(int)
	# position: 1-based
	if score_smoothing:
		# rolling
		conservation_scores = conservation_scores.sort_values(by=edited_pos_chr_col).reset_index(drop=True)

		win_size = 4
		half_win = (int(win_size) // 2)

		smoothed_scores = []

		positions = conservation_scores[edited_pos_chr_col].values
		scores = conservation_scores['score'].values

		for i, pos in enumerate(positions):
			start = pos - half_win	# inclusive
			end = pos + half_win	# inclusive
			
			mask = (positions >= start) & (positions <= end)
			window_scores = scores[mask]
			
			smoothed_scores.append(window_scores.mean() if len(window_scores) > 0 else float('nan'))

		conservation_scores['score'] = smoothed_scores

	df[edited_pos_chr_col] = df[edited_pos_chr_col].astype(int)
	df = pd.merge(df, conservation_scores, on=edited_pos_chr_col, how="left")

	if cons_subset != "all":
		df = df[df['Target region'].isin(cons_subset)]
	if isinstance(conservation_threshold, float):
		conservation_threshold = conservation_threshold
	elif conservation_threshold == "mean":
		# mean of selected regions
		conservation_threshold = df['score'].mean()
	elif conservation_threshold == "median":
		# median of selected regions
		conservation_threshold = df['score'].median()

	bins = [float('-inf'), conservation_threshold, float('inf')]
	# labels = [f'(-∞, {conservation_threshold:.2f})', f'[{conservation_threshold:.2f}, ∞)']
	labels = ["Other", "Conserved"]

	df['score_bin'] = pd.cut(df['score'], bins=bins, labels=labels, right=False)
	print(df['score_bin'].value_counts())

	# cdf
	plt.figure(figsize=(4,4))

	colors = {
		"Other": "#66CCFE",
		"Conserved": "#FF0066"
	}

	for label in labels:
		group_data = df[df['score_bin'] == label]['Average log2(CRISPR score)2'].sort_values()
		if len(group_data) == 0:
			continue
		cdf = np.arange(1, len(group_data)+1) / len(group_data)
		
		plt.plot(group_data, cdf, label=f'{conservation_method} {label}', color=colors[label])

	group_low = df[df['score_bin'] == 'Other']['Average log2(CRISPR score)2']
	median_low = group_low.median()
	group_high = df[df['score_bin'] == 'Conserved']['Average log2(CRISPR score)2']
	median_high = group_high.median()
	ks_stat, p_value = ks_2samp(group_low, group_high)


	if abs_LFC:
		plt.xlabel('abs(Average log2(CRISPR score)2)', fontsize=14)
	else:
		plt.xlabel('Average log$_2$(CRISPR score)', fontsize=14)
	plt.ylabel('CDF', fontsize=14)
	plt.title(f"{title}", fontsize=13)
	mant, exp = f"{p_value:.2e}".split("e")
	mant = float(mant); exp = int(exp)
	handles = [
		Line2D([], [], linestyle="None", marker=None, color=colors["Conserved"],
			label=f'Conserved (N={len(df[df.score_bin=="Conserved"]):,})'),
		Line2D([], [], linestyle="None", marker=None, color=colors["Other"],
			label=f'Other (N={len(df[df.score_bin=="Other"]):,})'),
		Line2D([], [], linestyle="None", marker=None, color="black",
			label=rf"$P={mant:.2f}\times 10^{{{exp}}}$"),
		Line2D([], [], linestyle="None", marker=None, color="black",
			label=f'Threshold: {conservation_method} {conservation_threshold:.2f}'),
	]
	plt.legend(handles=handles, loc="upper left", frameon=False, fontsize=13,
			handlelength=0, handletextpad=0, labelcolor="linecolor")
	plt.yticks([0, 0.5, 1], fontsize=13)
	plt.xticks(fontsize=13)
	xmin = df[df["score_bin"].isin(labels)]["Average log2(CRISPR score)2"].quantile(0.03)
	xmax = df[df["score_bin"].isin(labels)]["Average log2(CRISPR score)2"].quantile(0.97)
	plt.xlim(xmin, xmax)
	plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(0.5))
	# plt.axvline(x = 0, color='grey', linestyle="--", alpha=0.5)
	# plt.axhline(y = 0.5, color='grey', linestyle="--", alpha=0.5)
	plt.gca().spines['top'].set_visible(False)
	plt.gca().spines['right'].set_visible(False)
	plt.grid(False)
	plt.tight_layout()
	plt.savefig(f"/media/scratch/fy2306/projects/base_editing/plots/conservation/{conservation_method}_score_cdf.{title}.thresh{conservation_threshold}.pdf", 
			bbox_inches="tight",
			dpi=300,              
			transparent=True,
			format='pdf')
	plt.show()

In [ ]:
target_strand = "both"  # "both", "Top strand", "Bottom strand"
cons_subset = ['5\' UTR', 'Proximal promoter'] # 5\' UTR， Proximal promoter，Coding region
conservation_method = 'phyloP'
conservation_threshold = "median"
conservation_lfc_cdf(
	target_strand,
	cons_subset,
	conservation_method,
	conservation_threshold,
	title="RPS19 Promoter & 5' UTR",
	score_smoothing=True, abs_LFC=False)